# Day 77 — Project: chat with your docs

Build a grounded Q&A agent: chunk docs → embed → store → retrieve → answer **with
citations**. Retrieval runs offline (cheap_embed); the final answer uses the model. Swap
`cheap_embed` for [`shared.embed`](../../shared/llm.py) for real quality. > Needs a provider.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
import hashlib, math
from shared.llm import chat

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

KB = {
    "policy": "Employees get 25 days of paid leave per year.",
    "it": "Reset your password at portal.example.com/reset.",
    "travel": "Book travel at least 14 days in advance for approval.",
}
EMB = {k: cheap_embed(v) for k, v in KB.items()}

def ask(question, k=2):
    qe = cheap_embed(question)
    hits = sorted(((cosine(qe, e), id) for id, e in EMB.items()), reverse=True)[:k]
    context = "\n".join(f"[{id}] {KB[id]}" for _, id in hits)
    # TODO: call chat() with a system message telling it to answer from context and cite [ids],
    #       passing the question + context; return its reply
    raise NotImplementedError

# print(ask("How many vacation days do I get?"))

## 🔒 Solution

In [ ]:
import hashlib, math
from shared.llm import chat

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

KB = {
    "policy": "Employees get 25 days of paid leave per year.",
    "it": "Reset your password at portal.example.com/reset.",
    "travel": "Book travel at least 14 days in advance for approval.",
}
EMB = {k: cheap_embed(v) for k, v in KB.items()}

def ask(question, k=2):
    qe = cheap_embed(question)
    hits = sorted(((cosine(qe, e), id) for id, e in EMB.items()), reverse=True)[:k]
    context = "\n".join(f"[{id}] {KB[id]}" for _, id in hits)
    return chat([
        {"role": "system", "content": "Answer using ONLY the context. Cite the [id] you used. If unknown, say so."},
        {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"},
    ], temperature=0)

print(ask("How many vacation days do I get?"))